# Lab 6

You are tasked with evaluating card counting strategies for black jack. In order to do so, you will use object oriented programming to create a playable casino style black jack game where a computer dealer plays against $n$ computer players and possibily one human player. If you don't know the rules of blackjack or card counting, please google it. 

A few requirements:
* The game should utilize multiple 52-card decks. Typically the game is played with 6 decks.
* Players should have chips.
* Dealer's actions are predefined by rules of the game (typically hit on 16). 
* The players should be aware of all shown cards so that they can count cards.
* Each player could have a different strategy.
* The system should allow you to play large numbers of games, study the outcomes, and compare average winnings per hand rate for different strategies.

1. Begin by creating a classes to represent cards and decks. The deck should support more than one 52-card set. The deck should allow you to shuffle and draw cards. Include a "plastic" card, placed randomly in the deck. Later, when the plastic card is dealt, shuffle the cards before the next deal.

In [45]:
import random

# Represents a single playing card
class Card:

    # Initialize card with rank and suit
    def __init__(self, rank, suit):
        self.rank = rank
        self.suit = suit

    # Return blackjack value of the card
    def get_value(self):
        if self.rank == 'J' or self.rank == 'Q' or self.rank == 'K':
            return 10
        elif self.rank == 'A':
            return 11
        else:
            return int(self.rank)

    # Print card nicely
    def __str__(self):
        return self.rank + " of " + self.suit


# Represents a blackjack deck (multiple decks)
class Deck:

    # Initialize deck
    def __init__(self, num_decks):
        self.num_decks = num_decks
        self.cards = []
        self.cut_card_position = 0

        self.build_deck()
        self.shuffle()

    # Build full deck
    def build_deck(self):
        ranks = ['2','3','4','5','6','7','8','9','10','J','Q','K','A']
        suits = ['Hearts','Diamonds','Clubs','Spades']

        self.cards = []

        for d in range(self.num_decks):
            for suit in suits:
                for rank in ranks:
                    self.cards.append(Card(rank, suit))

        # Place cut card near end
        self.cut_card_position = random.randint(int(len(self.cards)*0.6), len(self.cards)-1)

    # Shuffle cards
    def shuffle(self):
        random.shuffle(self.cards)

    # Draw top card
    def draw_card(self):
        if len(self.cards) == 0:
            self.build_deck()
            self.shuffle()

        return self.cards.pop(0)

    # Check if reshuffle needed
    def needs_shuffle(self):
        if len(self.cards) <= self.cut_card_position:
            return True
        return False

2. Now design your game on a UML diagram. You may want to create classes to represent, players, a hand, and/or the game. As you work through the lab, update your UML diagram. At the end of the lab, submit your diagram (as pdf file) along with your notebook. 

In [46]:
# added pdf to lab 6 folder

3. Begin with implementing the skeleton (ie define data members and methods/functions, but do not code the logic) of the classes in your UML diagram.

In [47]:
# Represents a hand of cards
class Hand:

    def __init__(self):
        # Start with an empty list of cards
        self.cards = []

    def add_card(self, card):
        # Add a card to the hand
        pass

    def get_value(self):
        # Calculate the total blackjack value of the hand
        pass

    def is_bust(self):
        # Check if hand value exceeds 21 (bust)
        pass


# Base player class
class Player:

    def __init__(self, name, chips):
        # Store player's name and chips
        self.name = name
        self.chips = chips
        # Each player has a hand of cards
        self.hand = Hand()

    def bet(self):
        # Determine how many chips to bet (placeholder)
        pass

    def decide(self, visible_cards):
        # Decide whether to hit or stand given visible cards
        pass

    def win(self, amount):
        # Increase chips by amount won
        pass

    def lose(self, amount):
        # Decrease chips by amount lost
        pass

4. Complete the implementation by coding the logic of all functions. For now, just implement the dealer player and human player.

In [48]:
# Represents a hand of cards
class Hand:

    def __init__(self):
        # Start with empty card list
        self.cards = []

    def add_card(self, card):
        # Add card to hand
        self.cards.append(card)

    def get_value(self):
        # Compute blackjack hand total considering Aces as 1 or 11
        total = 0
        aces = 0

        for card in self.cards:
            total = total + card.get_value()
            if card.rank == 'A':
                aces = aces + 1

        # Adjust for Aces if total > 21
        while total > 21 and aces > 0:
            total = total - 10
            aces = aces - 1

        return total

    def is_bust(self):
        # Hand busts if value > 21
        return self.get_value() > 21


# Base player class
class Player:

    def __init__(self, name, chips):
        # Save player name and chips
        self.name = name
        self.chips = chips
        # Each player has a hand
        self.hand = Hand()

    def bet(self):
        # Always bet fixed 10 chips
        return 10

    def win(self, amount):
        # Add chips won
        self.chips = self.chips + amount

    def lose(self, amount):
        # Subtract chips lost
        self.chips = self.chips - amount

    def decide(self, visible_cards):
        # Hit if hand value is less than 17, else stand
        if self.hand.get_value() < 17:
            return True
        return False


# Dealer player with fixed decision rules
class Dealer(Player):

    def __init__(self):
        # Dealer has no chips, fixed name "Dealer"
        Player.__init__(self, "Dealer", 0)

    def decide(self, visible_cards):
        # Dealer hits on 16 or less, stands on 17 or more
        if self.hand.get_value() < 17:
            return True
        return False

5.  Test. Demonstrate game play. For example, create a game of several dealer players and show that the game is functional through several rounds.

In [49]:
# Controls the game flow
class Game:

    def __init__(self, players):
        # Players and dealer
        self.players = players
        self.dealer = Dealer()
        # Use 6 decks
        self.deck = Deck(6)
        # Cards visible to all players (for counting)
        self.visible_cards = []

    def play_round(self):

        # Reset hands for all players and dealer
        for p in self.players:
            p.hand = Hand()

        self.dealer.hand = Hand()
        self.visible_cards = []

        # Shuffle deck if cut card reached
        if self.deck.needs_shuffle():
            self.deck = Deck(6)

        # Deal 2 cards to each player and dealer
        for i in range(2):
            for p in self.players:
                card = self.deck.draw_card()
                p.hand.add_card(card)
                self.visible_cards.append(card)

            card = self.deck.draw_card()
            self.dealer.hand.add_card(card)
            self.visible_cards.append(card)

        # Players play in turn, hitting if they decide so
        for p in self.players:
            while not p.hand.is_bust() and p.decide(self.visible_cards):
                card = self.deck.draw_card()
                p.hand.add_card(card)
                self.visible_cards.append(card)

        # Dealer plays according to rules
        while self.dealer.decide(self.visible_cards):
            card = self.deck.draw_card()
            self.dealer.hand.add_card(card)
            self.visible_cards.append(card)

        # Determine results for each player
        dealer_value = self.dealer.hand.get_value()

        for p in self.players:
            bet = p.bet()

            if p.hand.is_bust():
                p.lose(bet)
            elif self.dealer.hand.is_bust():
                p.win(bet)
            elif p.hand.get_value() > dealer_value:
                p.win(bet)
            else:
                p.lose(bet)


# Test the game with two players for 5 rounds
players = [Player("P1", 100), Player("P2", 100)]
game = Game(players)

for i in range(5):
    game.play_round()
    print("Round", i+1)
    for p in players:
        print(p.name, p.chips)

Round 1
P1 90
P2 90
Round 2
P1 100
P2 100
Round 3
P1 90
P2 90
Round 4
P1 80
P2 80
Round 5
P1 90
P2 70


##### Original code (part I needed help on):
class Game:

    def __init__(self, players):
        # Players and dealer
        self.players = players
        self.dealer = Dealer()
        # Use 6 decks
        self.deck = Deck(6)
        # Cards visible to all players (for counting)
        self.visible_cards = []

    def play_round(self):

        # Reset hands for all players and dealer
        for p in self.players:
            p.hand = Hand()

        self.dealer.hand = Hand()
        self.visible_cards = []

        # Shuffle deck if cut card reached
        if self.deck.needs_shuffle():
            self.deck = Deck(6)

##### Prompt: Evaluate the code I have written and teach me how I can build on this code specifically adding methods to create a game of blackjack using elements and classes I currently have.

##### Explanation: I wasn't quite sure how to go about developing the game particularly from the dealing part and the part where players should take turns and after so I asked AI to help me understand how I can build on top of my code to incorporate the classes I have currently so I can learn and revise for the next parts in the lab and have updated my uml diagram too.

6. Implement a new player with the following strategy:

    * Assign each card a value: 
        * Cards 2 to 6 are +1 
        * Cards 7 to 9 are 0 
        * Cards 10 through Ace are -1
    * Compute the sum of the values for all cards seen so far.
    * Hit if sum is very negative, stay if sum is very positive. Select a threshold for hit/stay, e.g. 0 or -2.  

In [50]:
# Player that uses card counting strategy
class CountingPlayer(Player):

    def __init__(self, name, chips, threshold):
        # Initialize with base Player and counting threshold
        Player.__init__(self, name, chips)
        self.count = 0
        self.threshold = threshold

    def update_count(self, visible_cards):
        # Reset count before counting cards
        self.count = 0

        # Count cards based on values:
        # 2-6: +1, 7-9: 0, 10-Ace: -1
        for card in visible_cards:
            if card.rank in ['2', '3', '4', '5', '6']:
                self.count = self.count + 1
            elif card.rank in ['10', 'J', 'Q', 'K', 'A']:
                self.count = self.count - 1

    def decide(self, visible_cards):
        # Update count with visible cards
        self.update_count(visible_cards)

        # Hit if count is less than threshold (negative count)
        if self.count < self.threshold:
            return True
        return False

7. Create a test scenario where one player, using the above strategy, is playing with a dealer and 3 other players that follow the dealer's strategy. Each player starts with same number of chips. Play 50 rounds (or until the strategy player is out of money). Compute the strategy player's winnings. You may remove unnecessary printouts from your code (perhaps implement a verbose/quiet mode) to reduce the output.

In [51]:
def test_strategy():
    # Create counting player with threshold 0 and 3 other basic players
    counter = CountingPlayer("Counter", 100, 0)
    players = [counter, Player("P1", 100), Player("P2", 100), Player("P3", 100)]

    # Create game instance with players
    game = Game(players)

    # Play 50 rounds or until counter is out of chips (not implemented here)
    for i in range(50):
        game.play_round()

    # Show final chips for the counting player
    print("Final chips:", counter.chips)

test_strategy()

Final chips: -180


8. Create a loop that runs 100 games of 50 rounds, as setup in previous question, and store the strategy player's chips at the end of the game (aka "winnings") in a list. Histogram the winnings. What is the average winnings per round? What is the standard deviation. What is the probabilty of net winning or lossing after 50 rounds?


In [65]:
def simulate():
    # List to store final chips after each game
    results = []

    # Run 100 games
    for g in range(100):
        # Create the counting player and 3 normal players
        counter = CountingPlayer("Counter", 100, 0)
        players = [counter, Player("P1", 100), Player("P2", 100), Player("P3", 100)]
        game = Game(players)

        # Play 50 rounds per game
        for r in range(50):
            game.play_round()

        # Store the final chips of the counting player
        results.append(counter.chips)

    # Calculating the average chips
    total = 0
    for r in results:
        total += r
    average = total / len(results)
    print("Average chips:", average)
    print("Average per round:", (average - 100)/50)  # started with 100 chips, 50 rounds

    # Standard deviation calculation
    total_sq_diff = 0
    for r in results:
        total_sq_diff += (r - average)**2
    std_dev = (total_sq_diff / len(results))**0.5
    print("Standard deviation of chips:", std_dev)

    # Probability of net winning or losing
    net_win = 0
    net_loss = 0
    for r in results:
        if r > 100:  # net winning
            net_win += 1
        else:  # net losing or break-even
            net_loss += 1
    prob_win = net_win / len(results)
    prob_loss = net_loss / len(results)
    print("Probability of net winning:", prob_win)
    print("Probability of net losing:", prob_loss)

    # Histogramming the winnings
    min_chip = min(results)
    max_chip = max(results)
    bins = 10
    bin_size = (max_chip - min_chip) / bins
    counts = [0]*bins

    # Count results in each bin
    for r in results:
        index = int((r - min_chip) / bin_size)
        if index == bins:  # Include max in last bin
            index = bins - 1
        counts[index] += 1

    # Print histogram
    print("\nHistogram of final chips (approx.):")
    for i in range(bins):
        low = min_chip + i * bin_size
        high = low + bin_size
        bar = '*' * counts[i]  # each * represents one game
        print(f"{int(low):>4} - {int(high):>4}: {bar}")

# Run the simulation
simulate()

Average chips: -162.8
Average per round: -5.256
Standard deviation of chips: 64.99353814034136
Probability of net winning: 0.0
Probability of net losing: 1.0

Histogram of final chips (approx.):
-300 - -270: *
-270 - -240: *****
-240 - -210: ***********************
-210 - -180: ***********
-180 - -150: *******************
-150 - -120: ***********
-120 -  -90: ***************
 -90 -  -60: ****
 -60 -  -30: ********
 -30 -    0: ***


In [66]:
# Questions (outcome varies each time you run, but when I ran it, this is what I got)
# - Average chips: -162.8 → on average, the counting player lost 162.8 chips over 50 rounds
# - Average winnings per round: -5.256 → the player loses about 5.26 chips each round on average
# - Standard deviation: 64.99 → the results are moderately spread out, showing some variability in losses
# - Probability of net winning: 0.0 → the player never ended a game with more chips than they started
# - Probability of net losing: 1.0 → the player lost chips in every game
# - Histogram shows most final chips between -240 and -150, with very few smaller losses; no games ended in net profit
# Basically under this simulation, the card counting strategy is consistently losing and not profitable

9. Repeat previous questions scanning the value of the threshold. Try at least 5 different threshold values. Can you find an optimal value?

In [54]:
def test_thresholds():
    # Test thresholds from -4 to +4
    thresholds = [-4, -2, 0, 2, 4]

    for t in thresholds:
        total = 0

        # Run 50 games for each threshold
        for g in range(50):
            counter = CountingPlayer("Counter", 100, t)
            players = [counter, Player("P1", 100), Player("P2", 100), Player("P3", 100)]
            game = Game(players)

            # Play 50 rounds per game
            for r in range(50):
                game.play_round()

            total = total + counter.chips

        avg = total / 50
        print("Threshold", t, "Average:", avg)

test_thresholds()

Threshold -4 Average: -28.4
Threshold -2 Average: -87.6
Threshold 0 Average: -154.4
Threshold 2 Average: -264.8
Threshold 4 Average: -344.8


In [67]:
# - Tested 5 thresholds: -4, -2, 0, 2, 4
# - Average chips after 50 rounds for each threshold:
#     - Threshold -4 → -28.4 chips (smallest loss)
#     - Threshold -2 → -87.6 chips
#     - Threshold  0 → -154.4 chips
#     - Threshold  2 → -264.8 chips
#     - Threshold  4 → -344.8 chips (largest loss)
# - Interpretation:
#     - Lower thresholds (-4) performed best, resulting in the least losses
#     - Higher thresholds (2 or 4) performed worse, causing large losses
# - Optimal threshold: -4 → this threshold minimizes losses in this simulation

10. Create a new strategy based on web searches or your own ideas. Demonstrate that the new strategy will result in increased or decreased winnings. 

In [57]:
# Improved player with variable betting
class BetterCountingPlayer(CountingPlayer):

    # Increase bet when count is favorable
    def bet(self):
        if self.count > 2:
            # Bet more chips when count is high (good deck)
            return 20
        else:
            # Default bet otherwise
            return 10

In [58]:
# Simple test comparing both strategies
def compare_test():
    normal = CountingPlayer("Normal", 100, 0)
    better = BetterCountingPlayer("Better", 100, 0)

    game1 = Game([normal, Player("P1", 100), Player("P2", 100), Player("P3", 100)])
    game2 = Game([better, Player("P1", 100), Player("P2", 100), Player("P3", 100)])

    for i in range(50):
        game1.play_round()
        game2.play_round()

    print("Normal player chips:", normal.chips)
    print("Better player chips:", better.chips)

compare_test()

Normal player chips: -180
Better player chips: -240
